This notebook runs skeletonization on segmentation in a zarr format using the tensorstore package.

In [1]:
import skimage
import kimimaro
import cloudvolume
from cloudvolume import Skeleton
from scipy.spatial import KDTree
from joblib import Parallel, delayed, parallel_config, dump, load
import itertools
import scipy
import tarfile
import navis
import tarfile
from io import BytesIO
import concurrent.futures
import itertools
import numpy as np
from kimimaro.intake import merge
from matplotlib import pyplot as plt

from ac_segmentation.neurotorch.datasets.dataset import open_ZarrTensor, create_EmptyTensor
from ac_segmentation.reconnect_stack_navis import read_navis_neurons_tar, write_navis_skels_tar, write_kimi_skels_tar, swc_split_branches

In [ ]:
def label_binary_array(binary_arr, size_threshold=20):
    struct = scipy.ndimage.generate_binary_structure(3, 3)
    labeled_arr = np.empty(binary_arr.shape, dtype=np.uint32)
    num_features = scipy.ndimage.label(binary_arr, structure=struct, output=labeled_arr)

    if num_features > 1:
        labeled_arr = skimage.morphology.remove_small_objects(
            labeled_arr, min_size=size_threshold, connectivity=3, out=labeled_arr)

    return labeled_arr, num_features

def threshold_binarize_array(arr, threshold=0.2):
    return (arr >= threshold)

def skeletonize(out_arr, probability_threshold=0.2, label_size_threshold=50, scale=2, constant=5, 
                fill_holes=False, parallel=5, dust_threshold=10):
    # binarize volume, label, and skeletonize
    binary_arr = threshold_binarize_array(out_arr, threshold=probability_threshold)
    labeled_arr, num_feat = label_binary_array(binary_arr, size_threshold=label_size_threshold)
    skels = kimimaro.skeletonize(
        labeled_arr,
        teasar_params={
            "scale": scale, 
            "const": constant, # influences the finger branches allowed
            "pdrf_scale": 10000,
            "pdrf_exponent": 1,
            "soma_acceptance_threshold": 3500, # physical units
            "soma_detection_threshold": 750, # physical units
            "soma_invalidation_const": 300, # physical units
            "soma_invalidation_scale": 2,
            "max_paths": 50, # default None
        },
        dust_threshold=dust_threshold, # skip connected components with fewer than this many voxels
        anisotropy=(1,1,1), # default True #influences the dimension scale
        fix_branching=True, # default True
        fix_borders=True, # default True
        fill_holes=fill_holes, # default False
        fix_avocados=False, # default False
        progress=False, # default False, show progress bar
        parallel=parallel, # <= 0 all cpu, 1 single process, 2+ multiprocess
        parallel_chunk_size=100, # how many skeletons to process before updating progress bar
    )

    return skels

In [ ]:
def TS_skeletonize_volume(seg_arr, chunk_size=[1000,1000,1000], n_jobs=4, prob_thresh=0.2, label_size_threshold=20):
    def skel_chunk(start, end):
        skels = skeletonize(np.array(seg_arr[start[0]:end[0],start[1]:end[1],start[2]:end[2]]), probability_threshold=prob_thresh,label_size_threshold=label_size_threshold)

        if len(skels) != 0:
            skels = [skel for skid,skel in skels.items()]
            skels = Skeleton.simple_merge(skels).consolidate()
            skels.vertices += start
            return skels

    dx,dy,dz = seg_arr.shape
    xch, ych, zch = chunk_size
    sind_x, sind_y, sind_z = list(range(0,dx,xch)), list(range(0,dy,ych)), list(range(0,dz,zch))
    eind_x, eind_y, eind_z = [x + xch for x in sind_x], [x + ych for x in sind_y], [x + zch for x in sind_z]
    eind_x, eind_y, eind_z = [dx if ele > dx else ele for ele in eind_x], [dy if ele > dy else ele for ele in eind_y], [dz if ele > dz else ele for ele in eind_z]
    comb1 = list(itertools.product(sind_x,sind_y,sind_z))
    comb2 = list(itertools.product(eind_x,eind_y,eind_z))
    del sind_x, sind_y, sind_z, eind_x, eind_y, eind_z

    with parallel_config(backend="loky", inner_max_num_threads=2):
        results = Parallel(n_jobs=n_jobs)(delayed(skel_chunk)(x, y) for x, y in zip(comb1,comb2))

    #extract individual skeletons
    skels = [i for i in results if i is not None]
    out_skels = []
    for sk in skels:
        out_skels += sk.components()
        
    return out_skels

In [ ]:
#Run skeletonization
%time skels = TS_skeletonize_volume(arr, chunk_size=[1000,1000,1000], n_jobs=10, prob_thresh=.5, label_size_threshold=5)

#Write neurons to tar file as swcs
skel_dir = "/ACdata/Users/connorl/Skeletons/Out_Skels.swc.gz"
write_kimi_skels_tar(skel_dir, skels)